# Pré-processamento 3 — Maceió

Este notebook transforma os dados de ocorrências em uma matriz de série temporal:
1. Carrega o CSV com índice de célula do grid
2. Exploração inicial dos dados (anos disponíveis)
3. Calcula a hora do ano para cada ocorrência
4. Conta ocorrências por célula × hora (todos os tipos juntos), monta uma matriz completa e salva em um único CSV

**Entrada:** `maceio_with_grid_index.csv`

**Saída:** `maceio_all_crimes_grid.csv` (matriz célula × hora, contagem agregada de todos os tipos selecionados)


## 1. Carregamento dos dados com índice de célula

In [ ]:
# Carrega o CSV de Maceió com a coluna 'grid_cell_id' (índice da célula do grid)
# Converte DATA_HORA para datetime

import pandas as pd

df_crimes = pd.read_csv(
    "./output/maceio/maceio_with_grid_index.csv",
    delimiter=";",
    parse_dates=["DATA_HORA"]
)

# Valor de GRID_SIZE calculado no notebook pre-processing-1
GRID_SIZE = 73

## 2. Exploração dos dados

In [7]:
display(df_crimes)
df_crimes.info()
years = sorted(df_crimes['DATA_HORA'].dt.year.unique())
display(years)

,Unnamed: 0,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO,cell
0,0,-35.698388,-9.658217,street_robbery,2022-01-01 02:00:00,Maceió,2858
1,1,-35.703405,-9.660662,street_robbery,2022-01-01 02:00:00,Maceió,3004
2,2,-35.710142,-9.653007,vehicle_robbery,2022-01-01 02:00:00,Maceió,3151
3,3,-35.713147,-9.668955,street_robbery,2022-01-01 03:00:00,Maceió,3221
4,4,-35.703983,-9.663635,street_robbery,2022-01-01 04:00:00,Maceió,3003
...,...,...,...,...,...,...,...
72707,72707,-35.756315,-9.662037,street_robbery,2014-12-22 12:50:00,Maceió,4098
72708,72708,-35.715706,-9.652401,vehicle_robbery,2014-12-22 19:00:00,Maceió,3224
72709,72709,-35.755908,-9.586733,residential_robbery,2014-12-23 05:30:00,Maceió,4114
72710,72710,-35.738317,-9.624963,street_robbery,2014-12-23 09:30:00,Maceió,3741


<class 'pandas.DataFrame'>
RangeIndex: 72712 entries, 0 to 72711
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Unnamed: 0   72712 non-null  int64         
 1   LONGITUDE    72712 non-null  float64       
 2   LATITUDE     72712 non-null  float64       
 3   ROUBO_GROUP  72712 non-null  str           
 4   DATA_HORA    72712 non-null  datetime64[us]
 5   CIDADE_FATO  72712 non-null  str           
 6   cell         72712 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 3.9 MB


[np.int32(2012),
 np.int32(2013),
 np.int32(2014),
 np.int32(2015),
 np.int32(2016),
 np.int32(2017),
 np.int32(2018),
 np.int32(2019),
 np.int32(2020),
 np.int32(2021),
 np.int32(2022)]

## 4. Cálculo da hora do ano

Para cada ocorrência, calcula quantas horas se passaram desde o início do ano. Isso permite organizar os dados em uma grade temporal uniforme.

In [9]:
# Calcula a hora do ano para cada ocorrência
# Exemplo: 1 de janeiro 00:00 = hora 0, 2 de janeiro 00:00 = hora 24
# Utiliza pandarallel para processamento paralelo

import numpy as np
import datetime
from pandarallel import pandarallel
import os

cpu_count = os.cpu_count() or 1
nb_workers = max(1, min(cpu_count, 24))
pandarallel.initialize(nb_workers=nb_workers, progress_bar=True)


def hour_of_year(dt):
    beginning_of_year = datetime.datetime(
        dt["DATA_HORA"].year, 1, 1, tzinfo=dt["DATA_HORA"].tzinfo
    )
    return pd.Series(
        {
            "hour_year": (dt["DATA_HORA"] - beginning_of_year).total_seconds()
            // 3600
        }
    )


df_crimes
display("Translate date of crime to hour of the year")
df_hour = df_crimes.merge(
    df_crimes.parallel_apply(hour_of_year, axis=1), left_index=True, right_index=True
)
display("Initial shape: ", df_hour.shape)
display("Initial shape: ", df_hour)

INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


'Translate date of crime to hour of the year'

'Initial shape: '

(72712, 8)

'Initial shape: '

,Unnamed: 0,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO,cell,hour_year
0,0,-35.698388,-9.658217,street_robbery,2022-01-01 02:00:00,Maceió,2858,2.0
1,1,-35.703405,-9.660662,street_robbery,2022-01-01 02:00:00,Maceió,3004,2.0
2,2,-35.710142,-9.653007,vehicle_robbery,2022-01-01 02:00:00,Maceió,3151,2.0
3,3,-35.713147,-9.668955,street_robbery,2022-01-01 03:00:00,Maceió,3221,3.0
4,4,-35.703983,-9.663635,street_robbery,2022-01-01 04:00:00,Maceió,3003,4.0
...,...,...,...,...,...,...,...,...
72707,72707,-35.756315,-9.662037,street_robbery,2014-12-22 12:50:00,Maceió,4098,8532.0
72708,72708,-35.715706,-9.652401,vehicle_robbery,2014-12-22 19:00:00,Maceió,3224,8539.0
72709,72709,-35.755908,-9.586733,residential_robbery,2014-12-23 05:30:00,Maceió,4114,8549.0
72710,72710,-35.738317,-9.624963,street_robbery,2014-12-23 09:30:00,Maceió,3741,8553.0


## 5. Construção da matriz agregada de ocorrências

Para cada ano, conta o número de ocorrências por célula e hora do ano (somando todos os tipos selecionados em pre-processing). Gera uma matriz completa (célula × hora). Todas as células e horas sem ocorrência são preenchidas com NaN. Os anos são concatenados e o resultado é salvo em um único CSV.


In [ ]:
# Conta ocorrências de TODOS os tipos selecionados juntas (sem separar por ROUBO_GROUP)
# Para cada ano, gera matriz célula × hora_do_ano e concatena horizontalmente

df_all_years_list = []
for y in years:
    dfy = df_hour[df_hour["DATA_HORA"].dt.year == y].copy()

    # Conta ocorrências por célula e hora do ano (somando todos os tipos)
    counts = dfy.groupby(["grid_cell_id", "hour_year"]).size()

    # Linhas = células, colunas = horas do ano
    grid = counts.unstack("hour_year")

    # Total de horas no ano (considera bissexto)
    beginning_of_y1 = datetime.datetime(y, 1, 1)
    beginning_of_y2 = datetime.datetime(y + 1, 1, 1)
    num_hours_total = int((beginning_of_y2 - beginning_of_y1).total_seconds() // 3600)

    # Garante que todas as horas e células estão presentes (NaN onde não há dados)
    grid = grid.reindex(columns=range(num_hours_total))
    grid = grid.reindex(index=range(GRID_SIZE ** 2))

    # Renomeia colunas adicionando sufixo do ano
    df_year = grid.copy()
    df_year.columns = [f"{c}_{y}" for c in df_year.columns.tolist()]
    df_all_years_list.append(df_year)

# Concatena todos os anos lado a lado (linhas = células, colunas = todas as horas de todos os anos)
df_all = pd.concat(df_all_years_list, axis=1)
# Transpõe (linhas = horas, colunas = células) — formato esperado pelo pre-trainning
df_all = df_all.T
print("Shape final (horas × células):", df_all.shape)

df_all.to_csv("./output/maceio/maceio_all_crimes_grid.csv")
print("Salvo em ./output/maceio/maceio_all_crimes_grid.csv")
